# 00 — Course orientation

**Objectives.** Explain what CasADi does, verify the environment, and adopt the course's shape conventions. Prerequisites: Python, NumPy, and calculus.

CasADi records matrix-valued expression graphs, computes sparse exact derivatives, and exposes those graphs to numerical solvers. It is not a computer algebra system and an NLP solver generally finds a local solution, not a proof of a global optimum. We write vectors as columns: $x\in\mathbb R^{n\times1}$.


## Learning path and assumptions

**Prerequisites:** Python, NumPy arrays, and elementary calculus.

**Problem:** Separate graph construction, numerical evaluation, and optimization.

**Assumptions and conventions:** Column vectors retain shape (n, 1); examples use deterministic defaults.

**How to read the results:** Run cells top to bottom once, then predict the effect of a control before running it. The static site includes default outputs; solver-changing controls require JupyterLab.

### Editable lesson setup

This cell makes the checked-out repository available to the notebook without
installing it as a Python package. It searches the current directory and its
parents for `casadi_tutorial/`, adds that repository root to `sys.path`, and
enables IPython's `autoreload` extension. With autoreload enabled, changes to the
editable CSTR, UR5e, integration, plotting, and diagnostic modules are picked up
before later cells execute.

There is no mathematical transformation in this setup cell. Its role is to make
the relationship between notebook code and local source code explicit:

$$
\text{notebook import} \longrightarrow
\text{current repository source},
$$

rather than resolving an older installed wheel or editable package.


In [2]:
# Make repository-local helpers importable without installing a package.
import sys
from pathlib import Path

repository_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "casadi_tutorial").is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

# Autoreload picks up edits to tutorial models before each notebook cell runs.
ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Verify the numerical environment and matrix conventions

This cell confirms the CasADi version used throughout the course and
introduces `DM`, CasADi's numerical matrix type. CasADi treats vectors as
two-dimensional column matrices, so constructing a vector from three
values produces a shape of $3\times 1$, while transposition produces a
shape of $1\times 3$. The call to `full()` converts the CasADi matrix to
a dense NumPy array at the boundary of symbolic/numerical code.

The vector and its transpose are

$$
x = \begin{bmatrix}1\\2\\3\end{bmatrix}\in\mathbb{R}^{3\times 1},
\qquad
x^\mathsf{T} = \begin{bmatrix}1&2&3\end{bmatrix}
\in\mathbb{R}^{1\times 3}.
$$

Keeping this column-vector convention consistent prevents subtle shape
errors in Jacobians, dynamics functions, and optimization constraints.


In [3]:
import casadi as ca
import numpy as np

# The tutorials target one CasADi release so graph and solver behavior is reproducible.
print("CasADi version:", ca.__version__)
assert ca.__version__ == "3.7.2"

# DM is CasADi's numeric matrix type. A one-dimensional sequence becomes a column.
numeric_vector = ca.DM([1, 2, 3])
numpy_vector = np.asarray(numeric_vector.full()).ravel()
print(
    "DM shape:",
    numeric_vector.shape,
    "transpose shape:",
    numeric_vector.T.shape,
    "NumPy values:",
    numpy_vector,
)

CasADi version: 3.7.2
DM shape: (3, 1) transpose shape: (1, 3) NumPy values: [1. 2. 3.]


## Mental model and common mistake

Python executes once to *construct* a graph. A CasADi `Function` later evaluates that graph many times. Ordinary Python branching on a symbolic value therefore cannot work; use `ca.if_else` carefully and remember that nonsmooth expressions can challenge gradient-based solvers.

**Recap.** Graph construction, differentiation, and numerical solution are distinct phases. Exercise: list which phase reports a solver's return status. [Solution](../solutions/01_foundations.ipynb).


## Reproducibility and live controls

A seed fixes the sampled data, while solver tolerance governs numerical acceptance. Record both when reproducing a failure. The same default functions run headlessly in CI.

In [4]:
import importlib.util
import platform
print("Python:", platform.python_version())
print("Live widgets:", importlib.util.find_spec("ipywidgets") is not None)
print("Interactive plots:", importlib.util.find_spec("plotly") is not None)
print("Fixed-seed sample:", np.random.default_rng(7).normal(size=3))

Python: 3.13.16
Live widgets: True
Interactive plots: True
Fixed-seed sample: [ 0.00123015  0.29874554 -0.27413786]


**Checkpoint:** explain why a fixed seed does not require exactly the same iteration count on every machine. The capstones derive their own models; shared modules provide integration, plots, and diagnostics.

## Recap and independent practice

State the mathematical object constructed in this module, the numerical assumptions it requires, and one check that could reveal a wrong result. Change one experimental setting at a time and explain the observed change before continuing.